In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             classification_report, ConfusionMatrixDisplay, confusion_matrix)

VARIANT = 'occupancy'   # <-- єдина зміна для вашого варіанта
#шлях до даних
DATA = Path(r'C:\\Users\\vasil\\OneDrive\\Рабочий стол\\Artem.V\\data')
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 20)

if VARIANT == 'bean':   # <-- тут лишається 'bean', не чіпайте
    df = pd.read_csv(DATA / 'drybean.csv')
    target = 'Class'
    features = [c for c in df.columns if c not in ('Class', 'split')]
else:
    df = pd.read_csv(DATA / 'occupancy.csv')
    target = 'Occupancy'
    features = ['Temperature', 'Humidity', 'Light', 'CO2', 'HumidityRatio']

#дані вже розмічені
train, val, test = (df[df['split'] == s] for s in ('train', 'val', 'test'))
X_tr, y_tr = train[features], train[target]
X_va, y_va = val[features], val[target]
X_te, y_te = test[features], test[target]
print('train / validation / test:', len(train), len(val), len(test))

train / validation / test: 8143 2665 9752


In [ ]:
#1 Завдання 1:(а)	вкажіть свій варіант у VARIANT і запустіть першу клітинку; перегляньте дані (head(), describe());
display(df.head())
display(df.describe())
print('ознак:', len(features))
print('класів:', y_tr.nunique())

In [ ]:
#1 Завдання 1:(б)	порахуйте, скільки прикладів кожного класу в навчальній вибірці (value_counts()), і побудуйте графік розподілу класів;
print(y_tr.value_counts())
y_tr.value_counts().sort_index().plot(kind='bar', title='Розподіл класів', xlabel='Occupancy', ylabel='Кількість')
plt.show()

In [ ]:
#1 Завдання 1:(в)	порахуйте точність (accuracy) «лінивої» моделі, яка завжди відповідає найчастішим класом, на валідаційній вибірці (validation).
majority = y_tr.value_counts().idxmax()          
lazy_pred = np.full(len(y_va), majority)        
lazy_acc = accuracy_score(y_va, lazy_pred)
print('Найчастіший клас:', majority)
print('Лінивої моделі:', round(lazy_acc, 4))

In [ ]:
#1 Завдання:класи не є збалансованими, оскільки порожня кімната займає дуже значну частину класу

In [ ]:
#2 Завдання 2:(а)	навчіть DecisionTreeClassifier(max_depth=3, random_state=0) на навчальній вибірці (train);
tree3 = DecisionTreeClassifier(max_depth=3, random_state=0)
tree3.fit(X_tr, y_tr)
print('Дерево навчено, глибина:', tree3.get_depth())

In [104]:
#2 Завдання 2:(б)	виведіть точність (accuracy) на навчальній (train) і валідаційній (validation) вибірках;
print('train:', tree3.score(X_tr, y_tr), 'val:', tree3.score(X_va, y_va))

train: 0.9893159769126857 val: 0.9782363977485928


In [ ]:
#2 Завдання 2:(в)	намалюйте дерево (plot_tree) і виведіть його текстом (export_text).
plt.figure(figsize=(18, 8))
plot_tree(tree3, feature_names=features, class_names=['0', '1'], filled=True, fontsize=9)
plt.show()

In [ ]:
#2 Завдання 2:Перше питання яке поставило дерево було чи є рівень освітлення меншим або рівним певному значенню

In [ ]:
#3 Завдання 3:(а)	навчіть RandomForestClassifier(n_estimators=100, random_state=0) на навчальній вибірці (train);
forest = RandomForestClassifier(n_estimators=100, random_state=0)
forest.fit(X_tr, y_tr)
print('Навчанно:', len(forest.estimators_))

In [ ]:
#3 Завдання 3:(б)	для дерева і лісу порахуйте на валідаційній вибірці (validation) точність (accuracy), влучність (precision) і повноту (recall), усереднені по класах (average="macro");
for name, m in [('дерево', tree3), ('ліс', forest)]:
    p = m.predict(X_va)
    print(name, 'accuracy', round(accuracy_score(y_va, p), 4),
          'precision', round(precision_score(y_va, p, average='macro'), 4),
          'recall', round(recall_score(y_va, p, average='macro'), 4))

In [ ]:
#3 Завдання 3:(в)	побудуйте матриці помилок (confusion matrix) обох моделей.
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, (name, m) in zip(ax, [('Дерево', tree3), ('Ліс', forest)]):
    ConfusionMatrixDisplay.from_estimator(m, X_va, y_va, ax=a, colorbar=False); a.set_title(name)
plt.show()

In [ ]:
#3 Завдання 3:точнішою виявилася модель дерева рішень +-97% проти +-3% у лісу.У випадку лісу більше помилок з пропущена людина,а в дерев навпаки більше хобних тривог

In [ ]:
#4 Завдання 4:(а)	для дерева з max_depth від 1 до 20 порахуйте точність (accuracy) на навчальній (train) і валідаційній (validation) вибірках і побудуйте графік;
depths = list(range(1, 21))
tr_acc, va_acc = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_tr, y_tr)
    tr_acc.append(t.score(X_tr, y_tr))
    va_acc.append(t.score(X_va, y_va))

plt.figure(figsize=(8, 4.5))
plt.plot(depths, tr_acc, marker='o', label='train')
plt.plot(depths, va_acc, marker='o', label='validation')
plt.xlabel('max_depth'); plt.ylabel('accuracy'); plt.xticks(depths)
plt.legend(); plt.grid(alpha=0.3)
plt.show()

display(pd.DataFrame({'depth': depths, 'train': np.round(tr_acc, 4), 'val': np.round(va_acc, 4)}).set_index('depth').T)

In [ ]:
#4 Завдання 4:(б)	для лісу з n_estimators = 1, 5, 10, 25, 50, 100, 200, 300 порахуйте точність (accuracy) на валідаційній вибірці (validation) і побудуйте графік;
ns = [1, 5, 10, 25, 50, 100, 200, 300]
rf_va = [RandomForestClassifier(n_estimators=n, random_state=0).fit(X_tr, y_tr).score(X_va, y_va) for n in ns]
plt.plot(ns, rf_va, marker='o'); plt.xlabel('estimators');
plt.ylabel('accuracy'); plt.show()

In [ ]:
#4 Завдання 4:(в)	оберіть найкращу глибину дерева за валідаційною вибіркою (validation) і лише після цього один раз перевірте найкраще дерево і ліс на тестовій вибірці (test).
depths = range(1, 21)
dt_val = [DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_tr, y_tr).score(X_va, y_va) for d in depths]
best_d = list(depths)[int(np.argmax(dt_val))]
best_dt = DecisionTreeClassifier(max_depth=best_d, random_state=0).fit(X_tr, y_tr)
best_rf = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_tr, y_tr)
print(f"Найкраща глибина:{best_d}(val: {max(dt_val):.4f})")
print(f"Точність найкращого дерева:{best_dt.score(X_te, y_te):.4f}")
print(f"Точність лісу:{best_rf.score(X_te, y_te):.4f}")

In [ ]:
#4 Завдання 4:Починається з 4.Достатньо 50-100 дерев.

In [ ]:
#5 Завдання 5:(а)	побудуйте графік важливості ознак (feature_importances_) для дерева і лісу;
ig, ax = plt.subplots(1, 2, figsize=(11, 4))
pd.Series(tree3.feature_importances_, features).sort_values().plot(kind='barh', ax=ax[0], title='Дерево')
pd.Series(forest.feature_importances_, features).sort_values().plot(kind='barh', ax=ax[1], title='Ліс')
plt.show()

In [ ]:
#5 Завдання 5:(б)	визначте три найважливіші ознаки лісу;
print(pd.Series(forest.feature_importances_, features).nlargest(3))

In [ ]:
#5 Завдання 5:(в)	навчіть ще один ліс (100 дерев) зі зміненим набором ознак і порівняйте з повним: варіант bean - лише три найважливіші ознаки; варіант occupancy - усі ознаки, крім Light.
f2 = [f for f in features if f != 'Light']
forest2 = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_tr[f2], y_tr)
print('повний ліс val/test:', forest.score(X_va, y_va), forest.score(X_te, y_te))
print('без Light  val/test:', forest2.score(X_va[f2], y_va), forest2.score(X_te[f2], y_te))
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_estimator(forest, X_te, y_te, ax=ax[0], colorbar=False); ax[0].set_title('Повний ліс')
ConfusionMatrixDisplay.from_estimator(forest2, X_te[f2], y_te, ax=ax[1], colorbar=False); ax[1].set_title('Без Light')
plt.show()

In [ ]:
#6 Завдання 6:(а)	створіть щонайменше три власні вхідні приклади як pd.DataFrame з тими самими стовпцями, що й features (підказка: bean - візьміть медіани двох сортів, X_tr[y_tr == 'SIRA'].median(), і точки між ними; occupancy - задайте різні Light і CO2, решту - типові значення);
index = ['ніч, порожньо', 
        'світло є, CO2 низький', 
        'сутінки, CO2 високий',
        'яскраве світло, CO2 низький', 
        'світло і CO2 помірні', 
        'межа за Light']
new = pd.DataFrame({
    'Temperature': [20.5, 21.0, 22.0, 21.0, 21.5, 21.2],
    'Humidity': [25.0, 27.0, 29.0, 27.0, 28.0, 26.5],
    'Light': [0.0, 450.0, 250.0, 500.0, 300.0, 365.1],
    'CO2': [450.0, 430.0, 900.0, 430.0, 650.0, 500.0],
    'HumidityRatio': [0.0035, 0.0042, 0.0047, 0.0041, 0.0044, 0.0040]}, index=index)[features]
display(new)

In [ ]:
#6 Завдання 6:(б)	отримайте прогнози найкращого дерева і лісу;
forest = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_tr, y_tr)
depths = list(range(1, 21))
va_acc = [DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_tr, y_tr).score(X_va, y_va) for d in depths]
best_depth = depths[int(np.argmax(va_acc))]
best_tree = DecisionTreeClassifier(max_depth=best_depth, random_state=0).fit(X_tr, y_tr)

out = pd.DataFrame({'прогноз дерева': best_tree.predict(new),
                    'прогноз лісу': forest.predict(new)}, index=new.index)
display(out)

In [ ]:
#6 Завдання 6:(в)	виведіть ймовірності класів за лісом (predict_proba).
proba = forest.predict_proba(new)
out['P(0)'] = proba[:, 0].round(3)
out['P(1)'] = proba[:, 1].round(3)
display(pd.concat([new, out], axis=1))

In [ ]:
#6 Завдання 6:Дерево рішень послідовно перевіряє порогові правила, тому при яскравому освітленні воно враховує низький рівень CO2 і класифікує кімнату як порожню. 
# Ліс оцінює ситуацію спрощено, спираючись на голоси більшості дерев, які лише реагують на яскраве світло й через це помилково вважають, що в кімнаті хтось є.

In [106]:
#7 Завдання 7:(а)	Індекс Джині кореня дерева. З рисунка дерева із завдання 2 випишіть samples і value для кореня та двох його дітей. Для кожного вузла порахуйте частки класів p_k = value_k / samples і індекс Джині:

n, n_L, n_R = 8143, 6333, 1810
p0, p1 = 6414 / n, 1729 / n
G = 1 - (p0**2 + p1**2)
p0_L, p1_L = 6324 / n_L, 9 / n_L
G_L = 1 - (p0_L**2 + p1_L**2)
p0_R, p1_R = 90 / n_R, 1720 / n_R
G_R = 1 - (p0_R**2 + p1_R**2)
print(f"G = {G:.5f}, G_L = {G_L:.5f}, G_R = {G_R:.5f}")


G = 0.33449, G_L = 0.00284, G_R = 0.09450


In [ ]:
#7 Завдання 7:(б)	Влучність (precision) і повнота (recall) одного класу. З матриці помилок лісу (завдання 3) для класу SIRA (bean) або класу 1 (occupancy) випишіть: скільки правильних (на діагоналі), скільки всього прогнозів цього класу (сума стовпця), скільки справжніх об'єктів цього класу (сума рядка). Порахуйте: влучність = правильні / сума стовпця, повнота = правильні / сума рядка.


In [ ]:
#7 Завдання 7:в)	Бутстреп (bootstrap). Кожне дерево лісу вчиться на вибірці з n рядків, взятих з поверненням. Порахуйте ймовірність, що конкретний рядок навчальної вибірки жодного разу не потрапить у вибірку дерева, для вашого n:
